# Clustering Techniques: K-Means & Hierarchical

By the end of this notebook you will be able to:
1. Implement K-Means clustering in scikit-learn and choose k using the elbow method
2. Plot and interpret a dendrogram for hierarchical clustering
3. Apply `AgglomerativeClustering` and compare it to K-Means
4. Evaluate clusters using the silhouette score and adjusted Rand index

**Dataset:** 178 Italian wines described by 13 chemical properties (alcohol content, acidity, color intensity, etc.), each from one of 3 cultivars grown in the same region. We'll hide the cultivar labels and see whether our algorithms can recover them from chemistry alone.

---
## Setup & Data

In [ ]:
!uv add numpy pandas matplotlib seaborn scikit-learn scipy

In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

warnings.filterwarnings("ignore")

# Plotting style
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 150
%config InlineBackend.figure_format = "retina"

### Load the Wine dataset

**We're going to pretend we don't know which cultivar each wine came from.** Our job is to discover the natural groupings from the chemistry alone.

The 13 features include: alcohol, malic acid, ash, alcalinity of ash, magnesium, total phenols, flavanoids, nonflavanoid phenols, proanthocyanins, colour intensity, hue, OD280/OD315 of diluted wines, and proline.

In [ ]:
from sklearn.datasets import load_wine

# Load dataset
wine = load_wine()

X = pd.DataFrame(wine.data, columns=wine.feature_names)
y_true = wine.target  # 0, 1, 2  (cultivar ID)
cultivar_names = wine.target_names  # ['class_0', 'class_1', 'class_2']

print(f"Shape:           {X.shape}  ({X.shape[0]} wines, {X.shape[1]} features)")
print(f"Cultivar counts: {dict(zip(*np.unique(y_true, return_counts=True)))}")
print()
X.head()

In [ ]:
X.describe().round(2)

### Visual exploration with PCA

We can't plot 13 features on a 2D surface all at once. We'll use **Principal Component Analysis** (PCA) to reduce to 2 dimensions for visualisation only. This is just for exploration, the actual clustering will run on all 13 scaled features.

> PCA finds the two directions of maximum variance in the data. Think of it as finding the "best angle" to photograph a 13-dimensional object.

In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# Scale first, PCA is distance-based and sensitive to units
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)
var_exp = pca.explained_variance_ratio_

df_pca = pd.DataFrame(X_pca, columns=["PC1", "PC2"])

fig, ax = plt.subplots(figsize=(7, 5))
sns.scatterplot(data=df_pca, x="PC1", y="PC2", ax=ax)
ax.set_xlabel(f"PC1  ({var_exp[0] * 100:.1f}% variance)", fontsize=12)
ax.set_ylabel(f"PC2  ({var_exp[1] * 100:.1f}% variance)", fontsize=12)
ax.set_title(
    f"PCA Projection\n"
    f"{var_exp.sum() * 100:.1f}% of total variance captured",
    fontsize=12,
)
plt.tight_layout()
plt.show()

print("Question: How many clusters do you think you can see?")

---
## K-Means Clustering

### Choose k: the Elbow Method

We fit K-Means for a range of k values and plot **inertia** (total within-cluster sum of squares) against k.
- Inertia always decreases as k increases, more clusters means each point is closer to a centre
- We look for the **elbow**: the point where adding one more cluster gives diminishing returns

> The elbow isn't about the biggest absolute drop (that's always k=1→k=2). It's where the **rate of decrease slows sharply**.

In [ ]:
from sklearn.cluster import KMeans

k_range = range(1, 11)
inertias = []

for k in k_range:
    km = KMeans(n_clusters=k, n_init=10, random_state=42)
    km.fit(X_scaled)
    inertias.append(km.inertia_)

df_elbow = pd.DataFrame({"k": list(k_range), "Inertia": inertias})

fig, ax = plt.subplots(figsize=(7, 4))
sns.lineplot(data=df_elbow, x="k", y="Inertia", marker="o", linewidth=2.2, markersize=8, ax=ax)
ax.axvline(x=3, color="tomato", linestyle="--", linewidth=1.8, label="k = 3 (elbow)")
ax.set_xlabel("Number of Clusters (k)", fontsize=12)
ax.set_ylabel("Inertia", fontsize=12)
ax.set_title("Elbow Method", fontsize=13)
ax.legend(fontsize=11)
ax.set_xticks(list(k_range))
plt.tight_layout()
plt.show()

### Fit K-Means with k = 3

In [ ]:
kmeans = KMeans(
    n_clusters=3,
    n_init=10,  # run 10 times with different random seeds, keep best result
    random_state=42,
)
kmeans.fit(X_scaled)

km_labels = kmeans.labels_
km_centers = kmeans.cluster_centers_

print(f"Inertia:       {kmeans.inertia_:.2f}")
print(f"Cluster sizes: {dict(zip(*np.unique(km_labels, return_counts=True)))}")

### Visualise K-Means clusters

We project the cluster assignments back onto the PCA axes we computed earlier.

In [ ]:
def plot_clusters_pca(X_pca, labels, title, ax=None, show_centers=False, centers_pca=None):
    """Scatter of cluster assignments on the first two PCA components."""
    if ax is None:
        fig, ax = plt.subplots(figsize=(6, 4.5))

    df_plot = pd.DataFrame({
        "PC1": X_pca[:, 0],
        "PC2": X_pca[:, 1],
        "Cluster": [f"Cluster {l}" for l in labels],
    })
    sns.scatterplot(data=df_plot, x="PC1", y="PC2", hue="Cluster", ax=ax)

    if show_centers and centers_pca is not None:
        ax.scatter(
            centers_pca[:, 0],
            centers_pca[:, 1],
            marker="*",
            s=350,
            color="white",
            edgecolors="black",
            linewidths=1,
            zorder=5,
            label="Centroids",
        )
        ax.legend(fontsize=9)

    ax.set_xlabel("PC1", fontsize=10)
    ax.set_ylabel("PC2", fontsize=10)
    ax.set_title(title, fontsize=12)
    return ax


# Draw the K-Means result
km_centers_pca = pca.transform(km_centers)
ax = plot_clusters_pca(
    X_pca, km_labels, "K-Means (k=3)", show_centers=True, centers_pca=km_centers_pca
)
plt.tight_layout()
plt.show()

---
### Silhouette score

The elbow method suggested k=3. Let's verify with the **silhouette score**.

1. Calculate the silhouette score for the k=3 K-Means result
2. Loop over k = 2 to 8, fit K-Means each time, collect silhouette scores
3. Plot silhouette score vs k and identify the best k

Does the silhouette score agree with the elbow?

In [ ]:
from sklearn.metrics import silhouette_score

# Step 1: Silhouette score for k=3
sil_k3 = silhouette_score(X_scaled, km_labels)
print(f"Silhouette score for k=3: {sil_k3:.3f}")

# Step 2: Sweep k = 2 to 8
sil_scores = []
for k in range(2, 9):
    km_tmp = KMeans(n_clusters=k, n_init=10, random_state=42)
    labels_tmp = km_tmp.fit_predict(X_scaled)
    sil_scores.append(silhouette_score(X_scaled, labels_tmp))

# Step 3: Bar plot with seaborn
k_range_sil = list(range(2, 9))
df_sil = pd.DataFrame({"k": k_range_sil, "Silhouette Score": sil_scores})

fig, ax = plt.subplots(figsize=(7, 4))
bar_colors = ["tomato" if k == 3 else "steelblue" for k in k_range_sil]
sns.barplot(data=df_sil, x="k", y="Silhouette Score", palette=bar_colors, ax=ax)
ax.axhline(y=sil_k3, color="tomato", linestyle="--", linewidth=1.5, label=f"k=3: {sil_k3:.3f}")
ax.set_xlabel("Number of Clusters (k)", fontsize=12)
ax.set_ylabel("Silhouette Score", fontsize=12)
ax.set_title("Silhouette Score by k", fontsize=13)
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

best_k = k_range_sil[sil_scores.index(max(sil_scores))]
print(f"Best k by silhouette: {best_k}")
print("Both the elbow and silhouette agree on k=3")

---
## Hierarchical Clustering

### The Dendrogram

Instead of specifying k upfront, hierarchical clustering builds the full merge history and lets you **decide where to cut afterwards**.

**How to read the dendrogram:**
- **X-axis**: individual samples
- **Y-axis**: distance at which two clusters were merged: higher = more dissimilar
- **Horizontal cut**: draw a line at a chosen height → the number of vertical lines it crosses = number of clusters
- Look for the **largest vertical gap**, that's where the most natural break in the hierarchy is

In [ ]:
from scipy.cluster.hierarchy import dendrogram, linkage

METHOD = "ward"
Z = linkage(X_scaled, method=METHOD)

fig, ax = plt.subplots(figsize=(12, 4))
dendrogram(
    Z,
    ax=ax,
    leaf_font_size=0,  # hide labels
)
ax.axhline(y=13, color="tomato", linestyle="--", linewidth=1.8, label="Cut (3 clusters)")
ax.set_title(f"Dendrogram ({METHOD} linkage)", fontsize=13)
ax.set_xlabel("Samples", fontsize=11)
ax.set_ylabel("Distance", fontsize=11)
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

print("Notice the three large, clearly separated subtrees, a strong visual signal for k=3.")

### Fit AgglomerativeClustering with k = 3

In [ ]:
from sklearn.cluster import AgglomerativeClustering

hc = AgglomerativeClustering(n_clusters=3, linkage="ward")
hc_labels = hc.fit_predict(X_scaled)

print("Cluster sizes:")
for cluster, count in zip(*np.unique(hc_labels, return_counts=True)):
    print(int(cluster), "->", int(count))

### Visualise Hierarchical clusters

In [ ]:
ax = plot_clusters_pca(X_pca, hc_labels, "Hierarchical Clustering (Ward, k=3)")
plt.tight_layout()
plt.show()

sil_hc = silhouette_score(X_scaled, hc_labels)
print(f"Silhouette score (Ward, k=3): {sil_hc:.3f}")

---
### Task

Ward linkage minimises variance, but other methods make different assumptions about what "distance between clusters" means.

**Task:**
1. Refit `AgglomerativeClustering` using `linkage='complete'` (maximum distance between any two points in the clusters)
2. Plot the Ward and Complete results side-by-side on the PCA projection
3. Calculate and compare silhouette scores for both

**Discussion:** Why might Complete linkage produce different clusters? Which would you choose and why?

**Stretch:** Also try `linkage='single'`. What happens and why?

#### Solution *(instructor cell)*

In [ ]:
# Step 1: Fit with complete linkage
hc_complete = AgglomerativeClustering(n_clusters=3, linkage="complete")
hc_complete_labels = hc_complete.fit_predict(X_scaled)

# Step 2: Side-by-side plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))
plot_clusters_pca(X_pca, hc_labels, "Hierarchical: Ward", ax=ax1)
plot_clusters_pca(X_pca, hc_complete_labels, "Hierarchical: Complete", ax=ax2)
plt.tight_layout()
plt.show()

# Step 3: Compare silhouette scores
sil_ward = silhouette_score(X_scaled, hc_labels)
sil_complete = silhouette_score(X_scaled, hc_complete_labels)
print(f"Silhouette (Ward):     {sil_ward:.3f}")
print(f"Silhouette (Complete): {sil_complete:.3f}")
print()
print("Ward typically outperforms Complete: its variance-minimisation criterion")
print("produces more compact, balanced clusters.")
print()
# Stretch: single linkage
hc_single = AgglomerativeClustering(n_clusters=3, linkage="single")
hc_single_labels = hc_single.fit_predict(X_scaled)
sil_single = silhouette_score(X_scaled, hc_single_labels)
print(f"Silhouette (Single):   {sil_single:.3f}")
print("Cluster sizes: (Single)")
for cluster, count in zip(*np.unique(hc_single_labels, return_counts=True)):
    print(int(cluster), "->", int(count))
print('Single linkage "chaining": one cluster absorbs almost all points.')

---
## The Reveal

Now we unmask the true cultivar labels and measure how well our algorithms recovered them without ever seeing them during training.

### Side-by-side: algorithms vs. ground truth

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))


def reveal_scatter(X_pca, labels, title, ax, label_prefix="Cluster"):
    df = pd.DataFrame({
        "PC1": X_pca[:, 0],
        "PC2": X_pca[:, 1],
        "Group": [f"{label_prefix} {l}" for l in labels],
    })
    sns.scatterplot(
        data=df,
        x="PC1",
        y="PC2",
        hue="Group",
        s=65,
        alpha=0.8,
        edgecolor="white",
        linewidth=0.4,
        ax=ax,
    )
    ax.set_xlabel("PC1", fontsize=10)
    ax.set_ylabel("PC2", fontsize=10)
    ax.set_title(title, fontsize=12)


reveal_scatter(X_pca, km_labels, "K-Means (k=3)", axes[0])
reveal_scatter(X_pca, hc_labels, "Hierarchical: Ward (k=3)", axes[1])
reveal_scatter(X_pca, y_true, "True Cultivar Labels", axes[2], label_prefix="Cultivar")

plt.suptitle("Clustering Results vs. True Cultivar Labels", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

### Quantify accuracy: Adjusted Rand Index

The **Adjusted Rand Index (ARI)** measures agreement between two label assignments, corrected for chance.
- ARI = 1.0 → perfect agreement
- ARI = 0.0 → no better than random

> Note: cluster IDs are arbitrary. K-Means "Cluster 0" is not necessarily the same group as Hierarchical "Cluster 0". ARI handles this: it measures whether the *groupings* match, not the *numbers*.

In [ ]:
from sklearn.metrics import adjusted_rand_score

ari_km = adjusted_rand_score(y_true, km_labels)
ari_hc = adjusted_rand_score(y_true, hc_labels)

print("Adjusted Rand Index (vs. true cultivar labels)")
print(f"  K-Means (k=3):             {ari_km:.3f}")
print(f"  Hierarchical, Ward (k=3): {ari_hc:.3f}")
print()
print("1.0 = perfect  |  0.0 = random")

### Where did each algorithm go wrong?

In [ ]:
results = pd.DataFrame({
    "True cultivar": [f"Cultivar {i}" for i in y_true],
    "K-Means cluster": km_labels,
    "Hierarchical cluster": hc_labels,
})

print("K-Means: confusion vs. true labels:")
print(pd.crosstab(results["True cultivar"], results["K-Means cluster"], margins=True))
print()
print("Hierarchical: confusion vs. true labels:")
print(pd.crosstab(results["True cultivar"], results["Hierarchical cluster"], margins=True))

---
### Discussion

Look at the side-by-side plots, the ARI scores, and the confusion tables above, then discuss:

1. **Which algorithm recovered the true cultivar structure better?** Is the difference meaningful?

2. **Look at the confusion table** which cultivar was hardest to separate? Can you see why in the PCA plot?

3. **We got a high ARI without ever seeing the labels.** What does this tell you about the chemical differences between these wine cultivars?

4. **Real-world challenge:** In a real clustering problem like customer segmentation, you have no ground truth to compare against. Given only the silhouette score and the cluster visualisation, how would you decide whether your clusters are actually useful?

5. **Bonus:** If you had to recommend one algorithm to a colleague who needs to cluster a new wine dataset with 50,000 samples, which would you choose and why?

#### Solution *(instructor notes)*

1. **Which algorithm did better?** Both should score ARI ≈ 0.90+. K-Means and Ward hierarchical are very close on Wine, the dataset is clean enough that both handle it well. The difference is not meaningful in practice.

2. **Hardest cultivar to separate?** Cultivars 1 and 2 tend to overlap slightly in PCA space (they're the two right-side groups). Looking at the confusion table, a few Cultivar 1 points will be assigned to Cultivar 2 and vice versa, these sit in the boundary region between the two clusters.

3. **What does the high ARI mean?** The three cultivars have chemically distinct profiles, likely because of differences in the grape variety, soil, or winemaking process. The 13 chemical features (especially flavanoids, colour intensity, and proline) carry enough signal to separate them with no supervision at all.

4. **No ground truth: how to evaluate?** A few approaches:
   - **Silhouette score** objective metric for cluster compactness and separation
   - **Domain validation** do the clusters make business/domain sense? Can you give them meaningful names?
   - **Stability** do the same clusters appear if you re-run on a random subsample?
   - **Downstream utility** do the cluster labels improve a downstream model or business decision?

5. **50,000 samples → K-Means.** Hierarchical clustering is O(n²) in time and memory: 50,000 samples would be very slow and potentially infeasible. K-Means scales linearly with n and handles large datasets well. If you still want a dendrogram, you could hierarchically cluster on a representative sample (e.g. the K-Means centroids themselves).

---
## Summary

| | K-Means | Hierarchical (Ward) |
|---|---|---|
| **Specify k?** | Yes, before fitting | Yes, after viewing dendrogram |
| **Silhouette score** | *see GAP 1* | ~0.57 |
| **ARI vs. true labels** | 0.897 | 0.790 |
| **Speed** | Fast, scales to large n | Slow, O(n²) |
| **Deterministic?** | No, use `n_init ≥ 10` | Yes |

**Key takeaways:**
- Always **scale** features before clustering, distance methods are unit-sensitive
- Use **elbow + silhouette together** to choose k, neither is definitive alone. With Wine, both agree cleanly on k=3
- **Ward linkage** is the best default for hierarchical clustering. Single linkage suffers from chaining
- Both methods recovered three chemically distinct wine cultivars with ~90% accuracy from raw measurements, with no labels
- In practice: run both, compare metrics, and ask which result makes the most **domain sense**